# Lab 9

In [ ]:
from pathlib import Path
import random

import kagglehub
import pandas as pd
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
from ultralytics import YOLO
from IPython.display import display

## Descarga y preparacion del dataset

In [ ]:
DATA_DIR = (
    Path(kagglehub.dataset_download("notaashman/multiclass-animal-detection"))
    / "Multi-Class Animal Detection.v1-yolov8"
)

print("Path to dataset files:", DATA_DIR)

TRAIN_IMAGES = DATA_DIR / "train" / "images"
TRAIN_LABELS = DATA_DIR / "train" / "labels"

CLASS_NAMES = [
    "Buffalo",
    "Camel",
    "Cat",
    "Cheetah",
    "Cow",
    "Deer",
    "Dog",
    "Elephant",
    "Goat",
    "Gorilla",
    "Hippo",
    "Horse",
    "Lion",
    "Monkey",
    "Panda",
    "Rat",
    "Rhino",
    "Tiger",
    "Wolf",
    "Zebra",
]

## Exploracion de datos

In [ ]:
counts = {}
for split in ["train", "valid", "test"]:
    path = DATA_DIR / split / "images"
    counts[split] = len(list(path.glob("*.*"))) if path.exists() else 0

dist = pd.Series(counts, name="count")
print(dist.to_frame())

In [ ]:
population = list(TRAIN_IMAGES.glob("*.*"))

In [ ]:
k = min(20, len(population))
sample_images = random.sample(population, k)

n_cols = 5
n_rows = (k + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 3 * n_rows))

for ax, img_path in zip(axes.flatten(), sample_images):
    img = Image.open(img_path).convert("RGB")
    draw = ImageDraw.Draw(img)
    w, h = img.size

    label_path = TRAIN_LABELS / f"{img_path.stem}.txt"
    title_labels = set()
    if label_path.exists():
        for line in label_path.read_text().splitlines():
            cls, x_c, y_c, bw, bh = map(float, line.split())

            cx, cy = x_c * w, y_c * h
            bw_pix, bh_pix = bw * w, bh * h
            x1, y1 = cx - bw_pix / 2, cy - bh_pix / 2
            x2, y2 = cx + bw_pix / 2, cy + bh_pix / 2
            draw.rectangle([x1, y1, x2, y2], outline="red", width=2)
            title_labels.add(CLASS_NAMES[int(cls)])

    ax.imshow(img)
    ax.set_title(", ".join(sorted(title_labels)) or "Sin etiqueta")
    ax.axis("off")

for ax in axes.flatten()[k:]:
    ax.axis("off")

plt.suptitle("Muestras de Entrenamiento")
plt.tight_layout()
plt.show()

## Definicion YAML YOLO

In [ ]:
yaml_content = f"""
path: {DATA_DIR.absolute()}
train: train/images
val: valid/images
test: test/images

nc: {len(CLASS_NAMES)}
names: {CLASS_NAMES}
""".strip()
with open(DATA_DIR / "animal-detection.yaml", "w") as f:
    f.write(yaml_content)

## Entrenamiento modelo

In [ ]:
model_nano = YOLO("yolo11n.pt")
model_small = YOLO("yolo11s.pt")
model_large = YOLO("yolo11l.pt")

In [ ]:
r_nano = model_nano.train(
    data=DATA_DIR / "animal-detection.yaml",
    epochs=20,
    imgsz=640,
    batch=16,
    patience=5,
    name="animal_yolov11n",
)

r_small = model_small.train(
    data=DATA_DIR / "animal-detection.yaml",
    epochs=20,
    imgsz=640,
    batch=16,
    patience=5,
    name="animal_yolov11s",
)

r_large = model_large.train(
    data=DATA_DIR / "animal-detection.yaml",
    epochs=20,
    imgsz=640,
    batch=16,
    patience=5,
    name="animal_yolov11l",
)

## Evaluacion del modelo

In [ ]:
display(Image.open("runs/detect/animal_yolov11n/results.png"))

display(Image.open("runs/detect/animal_yolov11n/confusion_matrix.png"))

In [ ]:
display(Image.open("runs/detect/animal_yolov11s/results.png"))

display(Image.open("runs/detect/animal_yolov11s/confusion_matrix.png"))

In [ ]:
display(Image.open("runs/detect/animal_yolov11l/results.png"))

display(Image.open("runs/detect/animal_yolov11l/confusion_matrix.png"))

## Inferencia

In [ ]:
preds_nano = model_nano.predict(
    source=str(DATA_DIR / "test" / "images"),
    conf=0.25,
    save=True,
    name="animal_yolov11n_predict",
    verbose=False,
)

In [ ]:
files_nano = Path("runs/detect/animal_yolov11n_predict/").glob("*.jpg")
for img in random.sample(files_nano, 6):
    display(Image.open(img))

In [ ]:
preds_small = model_small.predict(
    source=str(DATA_DIR / "test" / "images"),
    conf=0.25,
    save=True,
    name="animal_yolov11s_predict",
    verbose=False,
)

In [ ]:
files_small = Path("runs/detect/animal_yolov11s_predict/").glob("*.jpg")
for img in random.sample(files_small, 6):
    display(Image.open(img))

In [ ]:
preds_large = model_large.predict(
    source=str(DATA_DIR / "test" / "images"),
    conf=0.25,
    save=True,
    name="animal_yolov11l_predict",
    verbose=False,
)

In [ ]:
files_large = Path("runs/detect/animal_yolov11l_predict/").glob("*.jpg")
for img in random.sample(files_large, 6):
    display(Image.open(img))

## Conclusiones

### Velocidad vs. Precisión

En general, **YOLO11n (nano)** entrena e infiere más rápido y consume menos memoria, pero suele perder algo de precisión frente a **YOLO11s (small)**.

- **YOLO11n (nano):** mejor para despliegue en edge/móvil, pruebas rápidas y hardware limitado (GPU pequeña o incluso CPU).
- **YOLO11s (small):** mejor equilibrio entre costo computacional y precisión para proyectos productivos básicos.
- **YOLO11m/l/x (medium/large/xlarge):** ganan precisión en datasets complejos (más clases, oclusiones, objetos pequeños), pero requieren más VRAM, más tiempo y mayor costo.

Regla práctica de elección:
- Si priorizas latencia/costo: nano o small.
- Si priorizas exactitud y tienes recursos: medium/large.
- Usa xlarge solo cuando la mejora de precisión justifique claramente el costo adicional.

### mAP50 vs. mAP50-95

- **mAP50** evalúa detecciones correctas con **IoU >= 0.50**. Es una métrica más permisiva: una caja puede estar "aproximadamente bien" y contar como acierto.
- **mAP50-95** promedia el mAP entre IoU de **0.50 a 0.95** (paso 0.05). Es más estricta y refleja mejor la calidad de localización fina de las cajas.

Interpretación conjunta:
- mAP50 alto y mAP50-95 bastante menor suele indicar que el modelo detecta objetos, pero sus cajas no son muy precisas.
- Si ambas son altas, el modelo detecta bien y además localiza con precisión.

### Hiperparámetros críticos

- **batch (tamaño de lote):**
  - Más alto: entrenamiento potencialmente más estable/rápido por época, pero sube mucho el uso de VRAM.
  - Más bajo: menor VRAM, pero entrenamiento más ruidoso y normalmente más lento.
- **imgsz (tamaño de imagen):**
  - Más grande (p. ej. 640 -> 960): mejora detección de objetos pequeños, pero aumenta tiempo y memoria de forma importante.
  - Más pequeño: más rápido/barato, con posible caída de precisión (sobre todo en objetos pequeños).
- **patience (early stopping):**
  - Bajo: detiene antes, ahorra tiempo, pero puede cortar mejoras tardías.
  - Alto: permite convergencia más completa, con riesgo de entrenar de más si ya no hay ganancia real.

Recomendación práctica: comenzar con `imgsz=640`, ajustar `batch` al máximo estable de tu GPU sin OOM, y usar `patience` moderada (5-15) según tamaño/ruido del dataset.

### Data Augmentation

Técnicas útiles para mejorar generalización en detección de animales:
- Volteos horizontal/vertical (cuando tenga sentido semántico).
- Escalado, traslación, rotaciones leves y shear.
- Cambios de brillo, contraste, saturación y tono.
- Blur, ruido y compresión JPEG para simular cámaras reales.
- Mosaic y MixUp/CutMix (con moderación) para aumentar diversidad espacial.
- Random crop y multi-scale training.

Buenas prácticas:
- Evitar aumentos extremos que generen ejemplos irreales.
- Medir impacto con validación (activar/desactivar por bloques).
- Si hay desbalance de clases, combinar augmentación con muestreo/estrategias por clase.